In [43]:
import duckdb
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
import os

In [44]:
WAREHOUSE_TEST = "./test_duckdb_data/gtftestha.duckdb"

con = duckdb.connect(WAREHOUSE_TEST)
con.execute("SET TimeZone='UTC';")

con.sql("SHOW TABLES").df()

,name
0,agency
1,calendar
2,calendar_dates
3,delays_with_support_columns
4,feed_info
5,routes
6,shapes
7,stop_times
8,stops
9,trips


In [37]:
con.sql("SELECT * FROM stops LIMIT 10").df()


,stop_id,stop_code,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,stop_timezone,wheelchair_boarding
0,1,ABBAT8,Abattoirs,43.717886,7.284703,None,0,place_101376,None,1
1,2,ABBAT4,Abattoirs,43.718399,7.284448,None,0,place_101376,None,1
2,3,ROSEL2,Abbaye de Roseland,43.693389,7.228426,None,0,place_096609,None,0
3,4,ROSEL8,Abbaye de Roseland,43.693219,7.228365,None,0,place_096609,None,0
4,5,ARSE4,Abbaye de Roseland / Napoléon III,43.688851,7.231361,None,0,place_096610,None,0
5,6,ACROB6,Risso,43.703344,7.280371,None,0,place_096612,None,1
6,8,AEPRM1,Aéroport / Promenade,43.666336,7.212338,None,0,place_101377,None,1
7,11,AEPRM2,Aéroport / Promenade,43.666523,7.212107,None,0,place_101377,None,1
8,12,ALBER3,Albert 1er,43.69585,7.269182,None,0,place_096616,None,0
9,13,ALBER8,Jardin d'Arménie,43.695717,7.2666,None,0,place_096617,None,0


In [51]:
# time reel
con.sql("""
CREATE OR REPLACE VIEW actual AS
SELECT
  tu.trip_id,
  tu.stop_sequence,
  CAST(tu.arrival_dt AS TIMESTAMP WITH TIME ZONE) AS arrival_dt_utc
FROM trips_updates tu
WHERE tu.arrival_dt IS NOT NULL;
""")

con.sql("SELECT * FROM actual LIMIT 10").df()

,trip_id,stop_sequence,arrival_dt_utc
0,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,1,2025-09-11 11:45:42+00:00
1,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,2,2025-09-11 11:45:55+00:00
2,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,3,2025-09-11 11:46:44+00:00
3,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,4,2025-09-11 11:47:48+00:00
4,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,5,2025-09-11 11:48:50+00:00
5,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,6,2025-09-11 11:50:14+00:00
6,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,7,2025-09-11 11:51:09+00:00
7,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,8,2025-09-11 11:51:46+00:00
8,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,9,2025-09-11 11:52:12+00:00
9,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,11,2025-09-11 11:53:45+00:00


In [52]:
# time theorique
con.sql("""
CREATE OR REPLACE VIEW sched AS
SELECT
  st.trip_id,
  CAST(st.stop_sequence AS BIGINT) AS stop_sequence,
  st.stop_id,
  st.arrival_time_sec
FROM stop_times st
WHERE st.arrival_time_sec IS NOT NULL;
""")

con.sql("SELECT * FROM sched LIMIT 10").df()

,trip_id,stop_sequence,stop_id,arrival_time_sec
0,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,0,21681,30600
1,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,1,21652,30900
2,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,2,21644,32100
3,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,3,21398,33300
4,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,0,21398,62400
5,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,1,21644,63600
6,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,2,21652,64800
7,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,3,21681,65100
8,3064033-C32_A_1_C3202_17:00-RESEAU2021-C32-Wee...,0,21681,61200
9,3064033-C32_A_1_C3202_17:00-RESEAU2021-C32-Wee...,1,21652,61500


In [53]:
#join entre deux tables

con.sql("""
CREATE OR REPLACE VIEW joined AS
SELECT
  a.trip_id,
  a.stop_sequence,
  s.stop_id,
  a.arrival_dt_utc,
  CAST(a.arrival_dt_utc AT TIME ZONE 'Europe/Paris' AS DATE) AS local_service_day,
  s.arrival_time_sec
FROM actual a
JOIN sched s
  ON a.trip_id = s.trip_id
 AND a.stop_sequence = s.stop_sequence;
""")

con.sql("SELECT * FROM joined LIMIT 10").df()


,trip_id,stop_sequence,stop_id,arrival_dt_utc,local_service_day,arrival_time_sec
0,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,0,21583,2025-09-11 10:31:26+00:00,2025-09-11,45060
1,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,1,21261,2025-09-11 10:31:26+00:00,2025-09-11,45120
2,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,2,21265,2025-09-11 10:32:05+00:00,2025-09-11,45120
3,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,3,21266,2025-09-11 10:32:31+00:00,2025-09-11,45120
4,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,4,21267,2025-09-11 10:32:58+00:00,2025-09-11,45180
5,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,5,21268,2025-09-11 10:33:25+00:00,2025-09-11,45240
6,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,6,21269,2025-09-11 10:34:01+00:00,2025-09-11,45240
7,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,7,21270,2025-09-11 10:34:23+00:00,2025-09-11,45240
8,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,8,21271,2025-09-11 10:34:56+00:00,2025-09-11,45300
9,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,9,21272,2025-09-11 10:35:35+00:00,2025-09-11,45360


In [81]:
# retard moyens reel - delay-min

con.sql("""
CREATE OR REPLACE VIEW per_event AS
SELECT
  trip_id,
  stop_sequence,
  stop_id,
  arrival_dt_utc,
  (
    (local_service_day + arrival_time_sec * INTERVAL '1 second')
    AT TIME ZONE 'Europe/Paris'
  ) AS scheduled_ts_utc,
  EXTRACT(
    EPOCH FROM (
      arrival_dt_utc - (
        (local_service_day + arrival_time_sec * INTERVAL '1 second')
        AT TIME ZONE 'Europe/Paris'
      )
    )
  ) / 60.0 AS delay_min
FROM joined;
""")

con.sql("SELECT * FROM per_event ORDER BY arrival_dt_utc DESC LIMIT 50").df()


,trip_id,stop_sequence,stop_id,arrival_dt_utc,scheduled_ts_utc,delay_min
0,4542749-44_R_99_4401_14:18-RESEAU2023-44-Semai...,7,4145,2025-09-11 12:44:41+00:00,2025-09-11 12:25:00+00:00,19.683333
1,4542693-44_A_50_4401_14:10-RESEAU2023-44-Semai...,7,4135,2025-09-11 12:33:42+00:00,2025-09-11 12:18:00+00:00,15.700000
2,6218561-84_R_95_8403_13:56-PROJET2025-84-Semai...,33,74,2025-09-11 12:30:32+00:00,2025-09-11 12:30:00+00:00,0.533333
3,6077772-58_A_50_5802_14:20-PROJET2025-58-Semai...,10,5162,2025-09-11 12:30:32+00:00,2025-09-11 12:31:00+00:00,-0.466667
4,4669793-62_A_46_6202_14:00-RESEAU2023-62-Semai...,28,6099,2025-09-11 12:30:27+00:00,2025-09-11 12:30:00+00:00,0.450000
5,6416089-21_R_99_2104_14:02-SETP2025-21-Semaine-40,20,4006,2025-09-11 12:30:25+00:00,2025-09-11 12:30:00+00:00,0.416667
6,6428320-60_A_50_6001_14:10-PROJET2025-60-Semai...,20,2546,2025-09-11 12:30:09+00:00,2025-09-11 12:30:00+00:00,0.150000
7,6368868-09_R_99_0904_13:55-PROJET2025-09-Semai...,19,4269,2025-09-11 12:30:09+00:00,2025-09-11 12:30:00+00:00,0.150000
8,6355833-18_A_50_1802_13:59-SETP2025-18-Semaine-39,26,376,2025-09-11 12:30:06+00:00,2025-09-11 12:30:00+00:00,0.100000
9,6446507-05_A_98_0509_14:03-SETP2025-05-L-Ma-J-...,21,1144,2025-09-11 12:30:00+00:00,2025-09-11 12:30:00+00:00,0.000000


In [79]:
con.execute("DROP VIEW IF EXISTS delays_with_support_columns;")
con.execute("DROP VIEW IF EXISTS per_event;")



test

In [82]:
# associer retards en utilisant la view insere dans duckdb (delays_with_support_columns)
con.sql("""
CREATE OR REPLACE VIEW stop_with_delay AS
SELECT
  s.stop_id,
  s.stop_name,
  s.stop_lat,
  s.stop_lon,
  ROUND(AVG(pe.delay_min), 2) AS avg_delay_min
FROM stops s
LEFT JOIN (
  SELECT * FROM per_event
  WHERE arrival_dt_utc::DATE = CURRENT_DATE
) pe
ON s.stop_id = pe.stop_id
GROUP BY s.stop_id, s.stop_name, s.stop_lat, s.stop_lon;
""")

con.sql("SELECT * FROM stop_with_delay ORDER BY avg_delay_min DESC LIMIT 10").df()

,stop_id,stop_name,stop_lat,stop_lon,avg_delay_min
0,5171,Sainte-Geneviève,43.695547,7.174875,41.01
1,5030,Corniche Bellevue,43.670837,7.184012,40.94
2,5050,Fahnestock Forêt,43.685251,7.17733,40.83
3,5108,Mauberts,43.688735,7.176725,40.74
4,5102,Li Maioun,43.680396,7.180559,40.73
5,5107,Mas de Montalou,43.683569,7.179403,40.66
6,5140,Rond Point Ravet,43.66875,7.184069,40.59
7,5047,Ecole Montaleigne,43.693593,7.175439,40.57
8,5085,Le Collet Rouge,43.697912,7.174152,40.50
9,5100,Les Tines,43.706529,7.171491,40.49


In [74]:
con.sql("""
SELECT 
  stop_id, 
  arrival_dt_utc,
  scheduled_ts_utc,
  delay_min
FROM per_event
WHERE arrival_dt_utc::DATE = CURRENT_DATE
ORDER BY delay_min DESC
LIMIT 50
""").df()


,stop_id,arrival_dt_utc,scheduled_ts_utc,delay_min
0,5171,2025-09-11 10:46:01+00:00,2025-09-11 09:18:00+00:00,88.016667
1,5050,2025-09-11 10:43:59+00:00,2025-09-11 09:16:00+00:00,87.983333
2,5030,2025-09-11 10:40:55+00:00,2025-09-11 09:13:00+00:00,87.916667
3,5088,2025-09-11 10:41:54+00:00,2025-09-11 09:14:00+00:00,87.900000
4,5028,2025-09-11 10:39:52+00:00,2025-09-11 09:12:00+00:00,87.866667
5,3106,2025-09-11 10:54:52+00:00,2025-09-11 09:27:00+00:00,87.866667
6,5254,2025-09-11 10:36:50+00:00,2025-09-11 09:09:00+00:00,87.833333
7,5253,2025-09-11 10:34:49+00:00,2025-09-11 09:07:00+00:00,87.816667
8,5143,2025-09-11 10:49:44+00:00,2025-09-11 09:22:00+00:00,87.733333
9,5102,2025-09-11 10:42:44+00:00,2025-09-11 09:15:00+00:00,87.733333


In [83]:
con.sql("""
SELECT
  j.trip_id,
  j.stop_sequence,
  j.stop_id,
  j.arrival_dt_utc,
  j.local_service_day,
  j.arrival_time_sec,
  ((local_service_day + arrival_time_sec * INTERVAL '1 second') AT TIME ZONE 'Europe/Paris') AS scheduled_ts_utc,
  arrival_dt_utc - ((local_service_day + arrival_time_sec * INTERVAL '1 second') AT TIME ZONE 'Europe/Paris') AS delay_duration
FROM joined j
WHERE arrival_dt_utc::DATE = CURRENT_DATE
ORDER BY delay_duration DESC
LIMIT 10;
""").df()


,trip_id,stop_sequence,stop_id,arrival_dt_utc,local_service_day,arrival_time_sec,scheduled_ts_utc,delay_duration
0,5925326-54_A_47_5402_10:58-PROJET2025-54-Semai...,23,5171,2025-09-11 10:46:01+00:00,2025-09-11,40680,2025-09-11 09:18:00+00:00,0 days 01:28:01
1,5925326-54_A_47_5402_10:58-PROJET2025-54-Semai...,19,5050,2025-09-11 10:43:59+00:00,2025-09-11,40560,2025-09-11 09:16:00+00:00,0 days 01:27:59
2,5925326-54_A_47_5402_10:58-PROJET2025-54-Semai...,11,5030,2025-09-11 10:40:55+00:00,2025-09-11,40380,2025-09-11 09:13:00+00:00,0 days 01:27:55
3,5925326-54_A_47_5402_10:58-PROJET2025-54-Semai...,13,5088,2025-09-11 10:41:54+00:00,2025-09-11,40440,2025-09-11 09:14:00+00:00,0 days 01:27:54
4,5925326-54_A_47_5402_10:58-PROJET2025-54-Semai...,8,5028,2025-09-11 10:39:52+00:00,2025-09-11,40320,2025-09-11 09:12:00+00:00,0 days 01:27:52
5,5925326-54_A_47_5402_10:58-PROJET2025-54-Semai...,31,3106,2025-09-11 10:54:52+00:00,2025-09-11,41220,2025-09-11 09:27:00+00:00,0 days 01:27:52
6,5925326-54_A_47_5402_10:58-PROJET2025-54-Semai...,5,5254,2025-09-11 10:36:50+00:00,2025-09-11,40140,2025-09-11 09:09:00+00:00,0 days 01:27:50
7,5925326-54_A_47_5402_10:58-PROJET2025-54-Semai...,4,5253,2025-09-11 10:34:49+00:00,2025-09-11,40020,2025-09-11 09:07:00+00:00,0 days 01:27:49
8,5925326-54_A_47_5402_10:58-PROJET2025-54-Semai...,16,5102,2025-09-11 10:42:44+00:00,2025-09-11,40500,2025-09-11 09:15:00+00:00,0 days 01:27:44
9,5925326-54_A_47_5402_10:58-PROJET2025-54-Semai...,28,5143,2025-09-11 10:49:44+00:00,2025-09-11,40920,2025-09-11 09:22:00+00:00,0 days 01:27:44


Il ya un decalage de environ 88mins environ, donc c'est a cause de l'erreur de ne pas avoir considere les bus qui partent apres minuit